In [2]:
!pip install -q -U "google-genai>=2.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


In [3]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [4]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

In [5]:
import numpy as np

In [6]:
MODEL_ID = "gemini-embedding-2"

In [7]:

text = ["Hello world"]
result = client.models.embed_content(model=MODEL_ID, contents=text)
[embedding] = result.embeddings

# Print just a part of the embedding to keep the output manageable
print(str(embedding.values)[:50], '... TRIMMED]')

print(len(embedding.values))

[-0.022945018, 0.010467435, 0.036357775, -0.016608 ... TRIMMED]
3072


In [8]:
def prepare_document(content, title=None):
  if title is None:
    tilte = "none"
  return f"title : {title} | text : {content}"

def prepare_query(query):
  return f"task : search result | query : {query}"

In [9]:
documents = [
    {
        "id": "doc1_domestic",
        "title": "Spanish General Tax Act (Domestic)",
        "text": "Spanish General Tax Act, Article 9: Any individual residing physically within Spanish territory for more than 90 days in a calendar year for employment or business activities is classified as a taxable resident and is subject to local income taxation."
    },
    {
        "id": "doc2_treaty",
        "title": "India-Spain DTAA Treaty (Bilateral)",
        "text": "Convention Between India and Spain for Avoidance of Double Taxation, Article 15(2): Remuneration derived by an Indian resident working in Spain is taxable only in India if the recipient is present in Spain for not exceeding 183 days in any 12-month period, provided the employer is non-resident."
    },
    {
        "id": "doc3_corporate",
        "title": "Corporate Registration Act",
        "text": "Spanish Corporate Enterprises Act: Companies incorporated under Spanish law must maintain registered office records and file corporate tax returns annually."
    }
]

tot_docs = 3

In [10]:
doc_embeddings = []

for it in range(tot_docs):
  result = client.models.embed_content(model=MODEL_ID, contents=prepare_document(documents[it]["text"], documents[it]["title"]))
  [embedding] = result.embeddings
  doc_embeddings.append(embedding.values)

In [11]:
print(doc_embeddings)

[[-0.0061476487, -0.007199601, -0.0212558, 0.0090235835, 0.0036932682, -0.006360811, -0.012556868, -0.0046191444, 0.027037835, -0.0485092, 0.0006816046, 0.006622773, -0.007125357, -0.00015769129, -0.006711485, 0.0008858136, 0.027789908, -0.002230407, -0.0016529738, 0.006021917, -0.011997073, -0.01957252, 0.023393014, 0.0122803915, -0.001790747, 0.0068111965, 0.00024128276, -0.034575544, 0.005560919, 0.10644403, -0.0059248763, -0.00699546, 0.0001700396, 0.014779424, 0.017482026, 0.033057984, -0.01639152, -0.0070324107, 0.02697123, -0.013666463, 0.0099017285, 0.01867402, 0.008042887, -0.0032027508, 0.01712163, 0.006643694, -0.01315166, -0.0016416451, 0.002070017, -0.011064223, 0.006702655, 0.0072526056, 0.0099594025, 0.0012539274, -0.0012510337, 0.001859272, 0.012339184, 0.018887328, 0.013148188, -0.0019995351, 0.0007144641, -0.02376524, -0.008868131, 0.030352263, 0.01663304, -0.0037703004, 0.0015448844, -0.01670596, -0.022525495, 0.006117262, -0.017824583, -0.0021560914, -0.011790019, 0

In [12]:
query = "I am an Indian remote worker planning to stay 110 days in Spain for a foreign company. Do I owe local income tax?"

In [13]:
query_embedding = []
result = client.models.embed_content(model=MODEL_ID, contents=prepare_query(query))
[embedding] = result.embeddings
query_embedding.append(embedding.values)

In [14]:
print(len(query_embedding))
print(len(doc_embeddings))
print(type(np.stack(doc_embeddings)))
print(len(np.stack(doc_embeddings)))


1
3
<class 'numpy.ndarray'>
3


In [15]:
scores = np.dot(np.stack(doc_embeddings), query_embedding[0])
print(scores)
for i, doc in enumerate(documents):
    print(f"[{doc['id']}] Score: {scores[i]:.4f} - {doc['title']}")

[0.76493177 0.78213572 0.65107255]
[doc1_domestic] Score: 0.7649 - Spanish General Tax Act (Domestic)
[doc2_treaty] Score: 0.7821 - India-Spain DTAA Treaty (Bilateral)
[doc3_corporate] Score: 0.6511 - Corporate Registration Act


In [20]:
# Assuming k = 2, doc1 & doc2 would serve as a context to the model
# Select top-2 chunks based on our scores
retrieved_context = f"""
[Source 1]: {documents[1]['text']}
[Source 2]: {documents[0]['text']}
"""

prompt = f"""
You are a legal tax assistant. Answer the user's question using ONLY the provided context below.
If there are rules or limits mentioned, state clearly whether the user owes tax.

Context:
{retrieved_context}

User Question:
{query}
"""

In [23]:
print(prompt)


You are a legal tax assistant. Answer the user's question using ONLY the provided context below.
If there are rules or limits mentioned, state clearly whether the user owes tax.

Context:

[Source 1]: Convention Between India and Spain for Avoidance of Double Taxation, Article 15(2): Remuneration derived by an Indian resident working in Spain is taxable only in India if the recipient is present in Spain for not exceeding 183 days in any 12-month period, provided the employer is non-resident.
[Source 2]: Spanish General Tax Act, Article 9: Any individual residing physically within Spanish territory for more than 90 days in a calendar year for employment or business activities is classified as a taxable resident and is subject to local income taxation.


User Question:
I am an Indian remote worker planning to stay 110 days in Spain for a foreign company. Do I owe local income tax?



In [27]:
interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input=f"{prompt}"
)

print(interaction.output_text)

Based on the provided context, the tax outcome depends on the two rules outlined:

* **Under Source 2 (Spanish General Tax Act, Article 9):** Because you plan to stay in Spain for **110 days** for employment (which exceeds the 90-day limit in a calendar year), you are classified as a taxable resident and are subject to local income taxation in Spain.
* **Under Source 1 (India-Spain Double Taxation Convention, Article 15(2)):** Because you are an Indian resident present in Spain for **110 days** (which does not exceed 183 days in a 12-month period) and work for a foreign, non-resident employer, your remuneration is taxable **only in India**. 

**Summary:** 
While Spanish local law (Source 2) classifies you as subject to local income tax for staying over 90 days, the Tax Convention (Source 1) specifies that your income is taxable **only in India** since your stay is under 183 days with a non-resident employer.
